# 2.2 Bronze

Reads from the landing views, profiles the raw data, and produces a clean, deduplicated Delta table.

**What this notebook covers:**
- Data profiling (null counts, duplicate detection)
- Deduplication with `ROW_NUMBER()` — picks the latest record per customer
- Writing a managed Delta table to the bronze schema

**Input:** `de_lab.landing.v_customers`  
**Output:** `de_lab.bronze.customers`

In [0]:
SHOW TABLES IN de_lab.landing;

## Profile

In [0]:
SELECT
  COUNT(*)                                              AS total_records,
  COUNT(DISTINCT customer_id)                          AS unique_ids,
  COUNT(CASE WHEN customer_id IS NULL THEN 1 END)      AS null_ids,
  COUNT(CASE WHEN email IS NULL THEN 1 END)            AS null_emails,
  COUNT(CASE WHEN phone IS NULL THEN 1 END)            AS null_phones
FROM de_lab.landing.v_customers;

In [0]:
%python
df = spark.table("de_lab.landing.v_customers")
display(df.describe())

## Null Records

In [0]:
SELECT *
FROM de_lab.landing.v_customers
WHERE customer_id IS NULL
LIMIT 20;

## Deduplication

Some customers appear more than once with slightly different values (e.g. updated phone numbers). The latest record by `created_timestamp` is the source of truth.

`ROW_NUMBER()` assigns rank 1 to the newest record per `customer_id`; filtering to `rn = 1` keeps exactly one row per customer and handles timestamp ties deterministically.

In [0]:
SELECT customer_id, COUNT(*) AS cnt
FROM de_lab.landing.v_customers
WHERE customer_id IS NOT NULL
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY cnt DESC
LIMIT 10;

In [0]:
CREATE OR REPLACE TABLE de_lab.bronze.customers AS
WITH ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY created_timestamp DESC) AS rn
  FROM de_lab.landing.v_customers
  WHERE customer_id IS NOT NULL
)
SELECT * EXCEPT (rn)
FROM ranked
WHERE rn = 1;

In [0]:
SELECT COUNT(*) AS bronze_customers FROM de_lab.bronze.customers;

In [0]:
SELECT * FROM de_lab.bronze.customers LIMIT 10;